In [26]:
import pandas as pd
file_path = r"C:\Users\9191h\OneDrive\0. Work\2. 국어 강사\0. 학생 관리\[중1] - 1학기 진도 사항\weekly_message\0627_토2.xlsx"

df = pd.read_excel(file_path)

df.head()

,날짜,이름,주테\n점수,독서1~3,독서4~6,비문학,문학,문법,19번,20번,과제,수업,전달 사항,최종문자
0,2026-06-27,가온이,79,1.0,NaN,1.0,"13, 14",18,NaN,NaN,NaN,NaN,"2학기가 시작되었습니다. 독서는 난도가 높은 『이기적 유전자』를 읽기 시작했으며, ...",▷ 가온이는 이번 주간 테스트에서 79점을 받았습니다.\n\n- 독서 : 4/5\n...
1,2026-06-27,근휘,79,1.0,NaN,1.0,13,17,NaN,2.0,근휘는 글쓰기 노트를 제출하지 않았습니다. 글쓰기는 서술형 평가를 연습하는 중요한 ...,NaN,"2학기가 시작되었습니다. 독서는 난도가 높은 『이기적 유전자』를 읽기 시작했으며, ...",▷ 근휘는 이번 주간 테스트에서 79점을 받았습니다.\n\n- 독서 : 4/5\n-...
2,2026-06-27,가령이,100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2학기가 시작되었습니다. 독서는 난도가 높은 『이기적 유전자』를 읽기 시작했으며, ...",▷ 가령이는 이번 주간 테스트에서 100점을 받았습니다.\n\n▷ 과 제 : 우수\...
3,2026-06-27,준서,75,NaN,NaN,NaN,14,"17, 18",1.0,4.0,NaN,NaN,"2학기가 시작되었습니다. 독서는 난도가 높은 『이기적 유전자』를 읽기 시작했으며, ...",▷ 준서는 이번 주간 테스트에서 75점을 받았습니다.\n\n- 독서 : 5/5\n-...
4,2026-06-27,우철이,66,1.0,2.0,1.0,12,18,1.0,2.0,NaN,NaN,"2학기가 시작되었습니다. 독서는 난도가 높은 『이기적 유전자』를 읽기 시작했으며, ...",▷ 우철이는 이번 주간 테스트에서 66점을 받았습니다.\n\n- 독서 : 2/5\n...


In [27]:
question_desc = {
    10: "소설의 구성 방식 중 '영웅의 일대기'에 관한 문제",
    11: "소설의 구성 방식 중 '영웅의 일대기'를 소설에 적용하는 문제",
    12: "'편집자적 논평'을 판단하는 문제",
    13: "소설 속에 나타난 '주인공의 행동'을 판단하는 문제",
    14: "소설 속에 나타난 '상황에 맞는 속담'을 찾는 문제",

    15: "직접 구성 성분을 분석하는 문제",
    16: "단어의 구성성분을 분석하고 파생어를 판단하는 문제",
    17: "단어의 직접 구성 성분을 분석하는 문제",
    18: "구성 성분이 동일한 단어를 찾는 문제"
}

In [28]:
def is_empty(value):
    return pd.isna(value) or str(value).strip() == ""


def to_int(value):
    if is_empty(value):
        return 0
    return int(float(value))


def parse_numbers(value):
    if is_empty(value):
        return []

    text = str(value).replace(" ", "")
    return [int(float(x)) for x in text.split(",") if x != ""]


def create_wrong_text(row):
    texts = []

    # 1~3번 독서
    r13 = to_int(row["독서1~3"])
    if r13 > 0:
        texts.append(f"이번 주에 읽어야 하는 책의 내용을 활용한 {r13}문제를 틀렸습니다.")

    # 4~6 독서 / 7~9 비문학 통합 처리
    reading = to_int(row["독서4~6"])
    nonfiction = to_int(row["비문학"])
    total = reading + nonfiction

    if total > 0:
        if reading > 0 and nonfiction > 0:
            area = "독서, 비문학"
        elif reading > 0:
            area = "독서"
        else:
            area = "비문학"

        texts.append(f"주어진 지문 안에서 근거를 찾는 문제({area}) 부분에서 {total}개를 틀렸습니다.")

    # 문학
    lit_nums = parse_numbers(row["문학"])
    if lit_nums:
        lit_parts = [f"{question_desc[n]} 1개" for n in lit_nums if n in question_desc]
        if lit_parts:
            texts.append("문학은 " + ", ".join(lit_parts) + "를 틀렸습니다.")

    # 문법
    gram_nums = parse_numbers(row["문법"])
    if gram_nums:
        gram_parts = [f"{question_desc[n]} 1개" for n in gram_nums if n in question_desc]
        if gram_parts:
            texts.append("문법은 " + ", ".join(gram_parts) + "를 틀렸습니다.")

    # 어휘
    vocab_parts = []

    if not is_empty(row["19번"]):
        vocab_parts.append("사자성어의 뜻을 찾는 1문제")

    wrong20 = to_int(row["20번"])
    if wrong20 > 0:
        vocab_parts.append(f"단어의 뜻을 찾는 {wrong20}문제")

    if vocab_parts:
        texts.append("어휘는 " + ", ".join(vocab_parts) + "를 틀렸습니다.")

    return " ".join(texts)

In [29]:
result_texts = []

for _, row in df.iterrows():
    base_text = str(row["최종문자"]).strip()
    wrong_text = create_wrong_text(row)

    if wrong_text:
        marker = "\n▷ 과 제"

        if marker in base_text:
            base_text = base_text.replace(
                marker,
                "\n→ " + wrong_text + "\n\n▷ 과 제",
                1
            )

    result_texts.append(base_text)

In [30]:
from datetime import datetime
from pathlib import Path

# 저장 폴더
save_dir = Path(r"C:\Users\9191h\OneDrive\0. Work\2. 국어 강사\0. 학생 관리\[중1] - 1학기 진도 사항\weekly_result")

# 폴더 없으면 자동 생성

save_dir.mkdir(parents=True, exist_ok=True)

# 현재 시간
now = datetime.now()

weekday_kr = ["월", "화", "수", "목", "금", "토", "일"][now.weekday()]

file_name = now.strftime(f"%Y-%m-%d_{weekday_kr}_%H시%M분") + ".txt"

# 최종 저장 경로
save_path = save_dir / file_name

# 학생별 문자 합치기
class_text = "\n\n------------------------------\n\n".join(result_texts)

# 저장
with open(save_path, "w", encoding="utf-8") as f:
    f.write(class_text)

print(f"완료: {save_path}")

완료: C:\Users\9191h\OneDrive\0. Work\2. 국어 강사\0. 학생 관리\[중1] - 1학기 진도 사항\weekly_result\2026-06-27_토_17시25분.txt
